In [18]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

In [19]:
matplotlib.use('tkagg')
%matplotlib auto
%matplotlib auto

Using matplotlib backend: tkagg
Using matplotlib backend: tkagg


In [20]:
## 注意：所有关节并联关节相关数据为解算后等效串联的结果，没有驱动器侧的原始并联关节数据
ppv210_joint_names = ["left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint", \
                    "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint", \
                    "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",  \
                    "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",\
                    "waist_yaw_joint", "waist_roll_joint","waist_pitch_joint", \
                    "head_yaw_joint", "head_pitch_joint",\
                    "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint", "left_elbow_pitch_joint", \
                    "left_wrist_yaw_joint", "left_wrist_pitch_joint", "left_wrist_roll_joint", \
                    "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint", "right_elbow_pitch_joint", \
                    "right_wrist_yaw_joint", "right_wrist_pitch_joint", "right_wrist_roll_joint"]
upper_joint_names = [   "head_yaw_joint", "head_pitch_joint",\
                        "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint", "left_elbow_pitch_joint", \
                        "left_wrist_yaw_joint", "left_wrist_pitch_joint", "left_wrist_roll_joint", \
                        "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint", "right_elbow_pitch_joint", \
                        "right_wrist_yaw_joint", "right_wrist_pitch_joint", "right_wrist_roll_joint"]
lower_joint_names = [
                        # "waist_yaw_joint", "waist_roll_joint","waist_pitch_joint", \
                        "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint", \
                        "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint", \
                        "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",  \
                        "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint"]

# 修改为你的数据文件夹路径
data_folder = "/home/fourier/GRX_humanoid/real data/"

## 0901 wbc lower 策略, 11号机。2.5代执行器+20减速比膝盖

In [21]:
# 工况：可根据高度指令和速度指令组合区分，包括如下几种：
# 高度0.68m,前进速度0.6m/s
# 高度0.68m,侧向速度0.6m/s
# 高度0.68m,旋转速度0.7rad/s
# 高度0.88m,前进速度1.0m/s
# 高度0.88m,旋转速度0.7rad/s
# 高度0.88m,侧向速度0.6m/s
path = data_folder + "2509_机器人下蹲与行走实物数据/ppv220wbc2_log_250901_160157.csv"
data = np.loadtxt(path, delimiter=",")
print(data.shape)

(9928, 48)


In [22]:
ac_dim = 12
time = data[:,0]
joint_pos = data[:,1:1+ac_dim]
joint_vel = data[:, 1+ac_dim:1+2*ac_dim]
joint_torque = data[:,1+2*ac_dim:1+3*ac_dim]
vel_cmd = data[:, 1+3*ac_dim:1+3*ac_dim+3]
ha_cmd = data[:, 1+3*ac_dim+3:1+3*ac_dim+5]
vel_est = data[:, 1+3*ac_dim+5:1+3*ac_dim+8]
hgt_est = data[:, 45:47]
joint_power = abs(joint_vel * joint_torque)

In [23]:
## 关节位置
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_pos[:, i], label=f'jPos_{lower_joint_names[i]}', color='purple')
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Position and Velocity Command')
plt.show()

## 关节速度
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_vel[:, i], label=f'jVel_{lower_joint_names[i]}', color='purple')
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Velocity and Velocity Command')
plt.show()

## 关节力矩
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_torque[:, i], label=f'jTor_{lower_joint_names[i]}', color='purple')
    ax.legend(loc= 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Torque and Velocity Command')
plt.show()

## 关节功率
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_power[:, i], label=f'jPow_{lower_joint_names[i]}', color='purple')
    ax.legend(loc= 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Power and Velocity Command')
plt.show()

## 总关节功率
joint_power_sum = joint_power.sum(axis=1)
plt.figure(figsize=(20,10))
plt.plot(joint_power_sum, label='Total Joint Power', color='purple')
plt.plot(vel_cmd[:, 0], label='Vx', linestyle='--')
plt.plot(vel_cmd[:, 1], label='Vy', linestyle='--')
plt.plot(vel_cmd[:, 2], label='Wz', linestyle='--')
plt.plot(ha_cmd[:, 0], label='Hgt', linestyle='--')
plt.legend()
plt.xlabel('Time Step')
plt.ylabel('Power (W)')
plt.title('Total Joint Power Over Time')
plt.grid()
plt.tight_layout()
plt.show()

## 0930 wbc lower 策略, 11号机。2.5代执行器+20减速比膝盖

In [10]:
path = data_folder + "2509_机器人下蹲与行走实物数据/ppv220wbc6_log_250930_153307.csv"
# path = "/home/fourier/GRX_humanoid/real data/ppv220wbc6_log_250930_172148.csv"
data = np.loadtxt(path, delimiter=",")
print(data.shape)

(2255, 49)


In [ ]:
ac_dim = 12
time = data[:,0]
joint_pos = data[:,1:1+ac_dim]
joint_vel = data[:, 1+ac_dim:1+2*ac_dim]
joint_torque = data[:,1+2*ac_dim:1+3*ac_dim]
vel_cmd = data[:, 1+3*ac_dim:1+3*ac_dim+3]
ha_cmd = data[:, 1+3*ac_dim+3:1+3*ac_dim+5]
vel_est = data[:, 1+3*ac_dim+5:1+3*ac_dim+8]
hgt_est = data[:, 45:48]
joint_power = abs(joint_vel * joint_torque)

In [ ]:
## 关节位置
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_pos[:, i], label=f'jPos_{lower_joint_names[i]}', color='purple')
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Position and Velocity Command')
plt.show()

## 关节速度
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_vel[:, i], label=f'jVel_{lower_joint_names[i]}', color='purple')
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Velocity and Velocity Command')
plt.show()

## 关节力矩
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_torque[:, i], label=f'jTor_{lower_joint_names[i]}', color='purple')
    ax.legend(loc= 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Torque and Velocity Command')
plt.show()

## 关节功率
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_power[:, i], label=f'jPow_{lower_joint_names[i]}', color='purple')
    ax.legend(loc= 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Power and Velocity Command')
plt.show()

## 总关节功率
joint_power_sum = joint_power.sum(axis=1)
plt.figure(figsize=(20,10))
plt.plot(joint_power_sum, label='Total Joint Power', color='purple')
plt.plot(vel_cmd[:, 0], label='Vx', linestyle='--')
plt.plot(vel_cmd[:, 1], label='Vy', linestyle='--')
plt.plot(vel_cmd[:, 2], label='Wz', linestyle='--')
plt.plot(ha_cmd[:, 0], label='Hgt', linestyle='--')
plt.legend()
plt.xlabel('Time Step')
plt.ylabel('Power (W)')
plt.title('Total Joint Power Over Time')
plt.grid()
plt.tight_layout()
plt.show()

## 0930 wbc 策略, 15号机。2.5代执行器+43减速比膝盖

In [14]:
path = data_folder + "2509_机器人下蹲与行走实物数据/wbc_vert14_log_250930_160752.csv"
# path = "/home/fourier/GRX_humanoid/real data/ppv220wbc6_log_250930_172148.csv"
data = np.loadtxt(path, delimiter=",")
print(data.shape)

(3312, 106)


In [15]:
ac_dim = 31
time = data[:,0]
joint_pos = data[:,1:1+ac_dim]
joint_vel = data[:, 1+ac_dim:1+2*ac_dim]
joint_torque = data[:,1+2*ac_dim:1+3*ac_dim]
vel_cmd = data[:, 1+3*ac_dim:1+3*ac_dim+3]
ha_cmd = data[:, 1+3*ac_dim+3:1+3*ac_dim+5]
vel_est = data[:, 1+3*ac_dim+5:1+3*ac_dim+8]
hgt_est = data[:, 102:105]
joint_power = abs(joint_vel * joint_torque)

In [17]:
## 关节位置
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_pos[:, i], label=f'jPos_{lower_joint_names[i]}', color='purple')
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Position and Velocity Command')
plt.show()

## 关节速度
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_vel[:, i], label=f'jVel_{lower_joint_names[i]}', color='purple')
    ax.legend(loc = 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Velocity and Velocity Command')
plt.show()

## 关节力矩
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_torque[:, i], label=f'jTor_{lower_joint_names[i]}', color='purple')
    ax.legend(loc= 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Torque and Velocity Command')
plt.show()

## 关节功率
fig, axs = plt.subplots(3, 4, figsize=(40, 20), sharex=True)
for i in range(12):
    ax = axs[i // 4, i % 4]
    ax.plot(joint_power[:, i], label=f'jPow_{lower_joint_names[i]}', color='purple')
    ax.legend(loc= 'upper left')
    ax2 = ax.twinx()
    ax2.plot(vel_cmd[:, 0], label=f'Vx', linestyle='--')
    ax2.plot(vel_cmd[:, 1], label=f'Vy', linestyle='--')
    ax2.plot(vel_cmd[:, 2], label=f'Wz', linestyle='--')
    ax2.plot(ha_cmd[:, 0], label=f'Hgt', linestyle='--')
    ax2.legend(loc = 'upper right')
plt.tight_layout()
plt.title('Lower Joint Power and Velocity Command')
plt.show()

## 总关节功率
joint_power_sum = joint_power.sum(axis=1)
plt.figure(figsize=(20,10))
plt.plot(joint_power_sum, label='Total Joint Power', color='purple')
plt.plot(vel_cmd[:, 0], label='Vx', linestyle='--')
plt.plot(vel_cmd[:, 1], label='Vy', linestyle='--')
plt.plot(vel_cmd[:, 2], label='Wz', linestyle='--')
plt.plot(ha_cmd[:, 0], label='Hgt', linestyle='--')
plt.legend()
plt.xlabel('Time Step')
plt.ylabel('Power (W)')
plt.title('Total Joint Power Over Time')
plt.grid()
plt.tight_layout()
plt.show()